# 03 — LoRA adaptation

**Purpose:** Train adapter-only versions of the same Qwen checkpoint; save logs, trained-parameter count, dataset hashes and adapter files.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/Urdu_Qwen_Research_Notebooks'), Path('/content/Urdu_Qwen_Research_Notebooks')]
ROOT = next((p for p in CANDIDATES if (p / 'project_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Unzip the entire project folder into /kaggle/working/ or /content/, then run this notebook. See README.md.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print('Project folder:', ROOT)

### Training choices

Default: `10 examples/class`, rank 8, seed 42, two epochs. After the pilot works, add `25,50` and seed repeats. All training uses only the training split. Validation is available for selection but the shared training function does not read the test set.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU on Kaggle/Colab. CPU fine-tuning is intentionally blocked here.'
TASK='sib200'; LANG='ur'
SIZES=[10]  # after pilot, [10,25,50]
RANKS=[8]  # after pilot, [8,16] at a fixed training size
SEEDS=[42] # after pilot, [42,123,456]
MODEL_OVERRIDE = None  # set to saved English anchor path for controlled forgetting
DRY_RUN = False
print('GPU:',torch.cuda.get_device_name(0))
for seed in SEEDS:
    for n in SIZES:
        for rank in RANKS:
            run_name = f'{TASK}_lora_{LANG}_n{n}_s{seed}_r{rank}' + ('_anchor' if MODEL_OVERRIDE else '')
            if (ROOT/'outputs/checkpoints'/run_name/'complete.json').exists():
                print('Already complete; skip:',run_name); continue
            if DRY_RUN: print('Would run:',run_name); continue
            r=train_run('lora',TASK,n,seed,run_name=run_name,lang=LANG,rank=rank,model_override=MODEL_OVERRIDE)
            print('FINISHED',r['run_id'],'minutes:',round(r['wall_seconds']/60,2),
                  'peak GiB:',r['peak_gpu_allocated_gib'],'trainable %:',r['trainable_pct'])

The checkpoint contains **LoRA adapters**, not a standalone copy of the full Qwen model. For inference the base model and adapter must both be loaded. Do not combine adapters from different base checkpoints.